In [ ]:
from cemp_software_settings import load_and_apply_settings
result = load_and_apply_settings()
print(result)


In [ ]:
import os
import subprocess
import re
import csv
from concurrent.futures import ThreadPoolExecutor, as_completed
from openbabel import openbabel
from openpyxl import Workbook
from openbabel import pybel
import shutil
import time
import pandas as pd
from rdkit import Chem


In [ ]:
mol_excel_file = "HTQC_global_reaction_descriptors.xlsx"

In [ ]:

MAX_TASKS = 2

In [ ]:

NPROC = 10
MEM = 20  

In [ ]:



os.makedirs('energy/neutral/success', exist_ok=True)
os.makedirs('energy/neutral/failure', exist_ok=True)


os.makedirs('energy/oxidation/success', exist_ok=True)
os.makedirs('energy/oxidation/failure', exist_ok=True)


os.makedirs('energy/reduction/success', exist_ok=True)
os.makedirs('energy/reduction/failure', exist_ok=True)

In [ ]:

opt_freq_path = "opt+freq/success"
neutral_energy_path = "energy/neutral"
oxidation_energy_path = "energy/oxidation"
reduction_energy_path = "energy/reduction"

In [ ]:
NEW_GAUSSIAN_KEYWORDS = "b3lyp/6-311+G(d,p) em=gd3bj scale=0.9682"

In [ ]:

def convert_out_to_gjf(out_file, gjf_file):
    
    subprocess.run(["obabel", "-i", "out", out_file, "-o", "gjf", "-O", gjf_file])

In [ ]:

def extract_charge_and_coordinates(gjf_file):
    with open(gjf_file, 'r') as file:
        lines = file.readlines()
    
    
    atom_line_pattern = re.compile(r'^\s*([A-Za-z]{1,2})\s+([-]?\d+\.\d+)\s+([-]?\d+\.\d+)\s+([-]?\d+\.\d+)\s*$')
    
    
    start_index = None
    for index, line in enumerate(lines):
        if atom_line_pattern.match(line):
            start_index = index - 1
            break
    
    
    final_index = None
    for index, line in enumerate(lines[start_index + 1 : ]):
        if atom_line_pattern.match(line) == False:
            final_index = index
            break
    
    
    charge_and_multiplicity_line = lines[start_index]
    coordinates_lines = lines[start_index + 1 : final_index]        
    
    return charge_and_multiplicity_line, coordinates_lines

In [ ]:

def get_charge_and_multiplicity(opt_freq_path, df):

    
    result_dict = {}

    
    for filename in os.listdir(opt_freq_path):
        if filename.endswith('.gjf'):
            
            name = os.path.splitext(filename)[0]

            
            row = df[df['Name'] == name]
            if row.empty:
                continue  

            
            smiles = row['SMILES'].values[0]

            
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                continue  

            
            mol = Chem.AddHs(mol)

            
            neutral_charge = sum(atom.GetFormalCharge() for atom in mol.GetAtoms())

            
            neutral_multiplicity = sum(atom.GetNumRadicalElectrons() for atom in mol.GetAtoms()) + 1

            
            oxidized_charge = neutral_charge + 1  
            oxidized_multiplicity = neutral_multiplicity - 1 if neutral_multiplicity > 1 else neutral_multiplicity + 1

            
            reduced_charge = neutral_charge - 1  
            reduced_multiplicity = neutral_multiplicity - 1 if neutral_multiplicity > 1 else neutral_multiplicity + 1

            
            charge_multiplicity_dict = {
                'neutral': f"{neutral_charge} {neutral_multiplicity}",
                'oxidized': f"{oxidized_charge} {oxidized_multiplicity}",
                'reduced': f"{reduced_charge} {reduced_multiplicity}"
            }

            
            result_dict[name] = charge_multiplicity_dict

    return result_dict

In [ ]:
df = pd.read_excel(mol_excel_file)

In [ ]:
result_dict = get_charge_and_multiplicity(opt_freq_path, df)

In [ ]:
result_dict

In [ ]:

def modify_and_append_to_gjf(gjf_file, new_keywords, chk_directory, charge_and_multiplicity_line, coordinates_lines):
    current_dir = os.getcwd()
    chk_file = os.path.join(current_dir, gjf_file.replace('.gjf', '.chk'))
    with open(gjf_file, 'w') as file:
        file.write(f"%nprocshared={NPROC}\n")
        file.write(f"%mem={MEM}GB\n")
        file.write(f'%chk={chk_file}\n')
        file.write(f'# {new_keywords}\n\n')
        file.write(f'{gjf_file} energy \n\n')
        file.write(f"{charge_and_multiplicity_line}\n")  
        file.writelines(coordinates_lines)  
        file.write('\n\n')

In [ ]:

def run_gaussian_and_categorize(gjf_file, success_dir, failure_dir):
    base_name = os.path.basename(gjf_file).replace(".gjf", "")
    out_file = base_name + ".out"
    chk_file = base_name + ".chk"
    
    
    try:
        subprocess.run(["g16", gjf_file, out_file])

        
        with open(out_file, 'r') as file:
            contents = file.read()
            if "Normal termination" in contents:
                
                os.rename(gjf_file, os.path.join(success_dir, os.path.basename(gjf_file)))
                os.rename(out_file, os.path.join(success_dir, out_file))
                os.rename(chk_file, os.path.join(success_dir, chk_file))
                return out_file, True
            else:
                
                os.rename(gjf_file, os.path.join(failure_dir, os.path.basename(gjf_file)))
                os.rename(out_file, os.path.join(failure_dir, out_file))
                os.rename(chk_file, os.path.join(failure_dir, chk_file))
    except subprocess.CalledProcessError:
        
        os.rename(gjf_file, os.path.join(failure_dir, gjf_file))
        if os.path.exists(out_file):
            os.rename(out_file, os.path.join(failure_dir, out_file))
        if os.path.exists(chk_file):
            os.rename(chk_file, os.path.join(failure_dir, chk_file))

    return None, False    

In [ ]:
def check_and_move_files(energy_path, success_dir, failure_dir):
    
    
    for file in os.listdir(energy_path):
        if file.endswith(".out"):  
            file_path = os.path.join(energy_path, file)
            with open(file_path, 'r') as f:
                content = f.read()
                
                if "Normal termination" in content:
                    
                    target_dir = success_dir
                else:
                    
                    target_dir = failure_dir
                
                
                base_filename = os.path.splitext(file)[0]
                for ext in ['.gjf', '.chk', '.out']:
                    src_file = os.path.join(energy_path, base_filename + ext)
                    if os.path.exists(src_file):  
                        dst_file = os.path.join(target_dir, base_filename + ext)
                        os.rename(src_file, dst_file)

In [ ]:


for out_file in os.listdir(opt_freq_path):
    if out_file.endswith(".out"):
        
        name = os.path.splitext(out_file)[0]
        
        
        if name in result_dict:
            
            state_dict = result_dict[name]
            
            
            source_file = os.path.join(opt_freq_path, out_file)
            
            
            for state, charge_and_multiplicity_line in state_dict.items():
                
                if state == 'neutral':
                    target_file = os.path.join(neutral_energy_path, out_file.replace(".out", ".gjf"))
                    energy_path_state = neutral_energy_path
                elif state == 'oxidized':
                    target_file = os.path.join(oxidation_energy_path, out_file.replace(".out", ".gjf"))
                    energy_path_state = oxidation_energy_path
                elif state == 'reduced':
                    target_file = os.path.join(reduction_energy_path, out_file.replace(".out", ".gjf"))
                    energy_path_state = reduction_energy_path
                else:
                    continue  

                
                convert_out_to_gjf(source_file, target_file)

                
                _, coordinates_lines = extract_charge_and_coordinates(target_file)

                
                modify_and_append_to_gjf(
                    target_file,
                    NEW_GAUSSIAN_KEYWORDS,
                    energy_path_state,
                    charge_and_multiplicity_line,
                    coordinates_lines
                )

In [ ]:

energy_paths = [neutral_energy_path, oxidation_energy_path, reduction_energy_path]


def create_success_failure_dirs(energy_path):
    success_dir = os.path.join(energy_path, "success")
    failure_dir = os.path.join(energy_path, "failure")
    os.makedirs(success_dir, exist_ok=True)
    os.makedirs(failure_dir, exist_ok=True)
    return success_dir, failure_dir


def process_energy_path(energy_path):
    
    success_dir, failure_dir = create_success_failure_dirs(energy_path)
    
    
    with ThreadPoolExecutor(max_workers=MAX_TASKS) as executor:
        futures = [executor.submit(run_gaussian_and_categorize, os.path.join(energy_path, f), success_dir, failure_dir)
                   for f in os.listdir(energy_path) if f.endswith(".gjf")]
        for future in as_completed(futures):
            pass  
    
    
    check_and_move_files(energy_path, success_dir, failure_dir)


with ThreadPoolExecutor(max_workers=MAX_TASKS) as executor:
    futures = [executor.submit(process_energy_path, energy_path) for energy_path in energy_paths]
    for future in as_completed(futures):
        pass  